<a href="https://colab.research.google.com/github/Rksingh9883122/Bias-fairness/blob/MLOPs_VNIT/Bias_%26_Fairness_Machine_Learning_Algorithm_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install aif360
!pip install 'aif360[Reductions]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.0/240.0 kB 6.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing import Reweighing
from aif360.algorithms.inprocessing import AdversarialDebiasing
from aif360.algorithms.postprocessing import EqOddsPostprocessing
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

# Load the dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
column_names = [
    "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
    "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
    "hours-per-week", "native-country", "income"
]
df = pd.read_csv(url, header=None, names=column_names, na_values=" ?", sep=',\s', engine='python')

# Drop rows with missing values
df = df.dropna()

print(df)

# Target variable and protected attribute
df['income'] = df['income'].apply(lambda x: 1 if x == '>50K' else 0)
protected_attribute = 'sex'
df[protected_attribute] = df[protected_attribute].apply(lambda x: 1 if x == 'Male' else 0)

# Separate features and target
X = df.drop(columns=['income'])
y = df['income']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Preprocessing pipeline
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numeric_features = X.select_dtypes(exclude=['object']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])

# Fit and transform the training data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Convert to DataFrame for AIF360 compatibility
train_df = pd.DataFrame(X_train_processed.toarray(), columns=[f'feat_{i}' for i in range(X_train_processed.shape[1])])
train_df['income'] = y_train.values
train_df[protected_attribute] = X_train[protected_attribute].values

test_df = pd.DataFrame(X_test_processed.toarray(), columns=[f'feat_{i}' for i in range(X_test_processed.shape[1])])
test_df['income'] = y_test.values
test_df[protected_attribute] = X_test[protected_attribute].values

# Creating BinarylabelDataset Object for AIF360
train_dataset = BinaryLabelDataset(favorable_label=1, unfavorable_label=0,
                                   df=train_df,
                                   label_names=['income'],
                                   protected_attribute_names=[protected_attribute])

test_dataset = BinaryLabelDataset(favorable_label=1, unfavorable_label=0,
                                  df=test_df,
                                  label_names=['income'],
                                  protected_attribute_names=[protected_attribute])

# Analyze data bias
metric = BinaryLabelDatasetMetric(train_dataset, privileged_groups=[{protected_attribute: 1}], unprivileged_groups=[{protected_attribute: 0}])
print("Difference in mean outcomes between privileged and unprivileged groups:", metric.mean_difference())


pip install 'aif360[inFairness]'
Instructions for updating:
non-resource variables are not supported in the long term


       age         workclass  fnlwgt   education  education-num  \
0       39         State-gov   77516   Bachelors             13   
1       50  Self-emp-not-inc   83311   Bachelors             13   
2       38           Private  215646     HS-grad              9   
3       53           Private  234721        11th              7   
4       28           Private  338409   Bachelors             13   
...    ...               ...     ...         ...            ...   
32556   27           Private  257302  Assoc-acdm             12   
32557   40           Private  154374     HS-grad              9   
32558   58           Private  151910     HS-grad              9   
32559   22           Private  201490     HS-grad              9   
32560   52      Self-emp-inc  287927     HS-grad              9   

           marital-status         occupation   relationship   race     sex  \
0           Never-married       Adm-clerical  Not-in-family  White    Male   
1      Married-civ-spouse    Exec-manag

## **Pre-Processing**

In [ ]:
# Apply reweighing
RW = Reweighing(unprivileged_groups=[{protected_attribute: 0}], privileged_groups=[{protected_attribute: 1}])
train_dataset_transf = RW.fit_transform(train_dataset)

# Check new weights
print("Transformed dataset weights:", np.unique(train_dataset_transf.instance_weights, return_counts=True))


Transformed dataset weights: (array([0.78522587, 0.85149376, 1.09501191, 2.22115181]), array([ 4668,  6751, 10552,   821]))


## **In Processing**

In [ ]:
# Reset the TensorFlow graph
tf.reset_default_graph()

# Set up the TensorFlow session
sess = tf.Session()

# Set up the model
debiased_model = AdversarialDebiasing(privileged_groups=[{protected_attribute: 1}], unprivileged_groups=[{protected_attribute: 0}],
                                      scope_name='debiased_classifier', debias=True, sess=sess)
debiased_model.fit(train_dataset_transf)
predictions = debiased_model.predict(test_dataset)
# Ensure predictions and dataset do not contain invalid values
def sanitize_data(dataset):
    dataset.scores = np.nan_to_num(dataset.scores, nan=0.0, posinf=0.0, neginf=0.0)
    dataset.labels = np.nan_to_num(dataset.labels, nan=0.0, posinf=0.0, neginf=0.0)
    dataset.instance_weights = np.nan_to_num(dataset.instance_weights, nan=1.0, posinf=1.0, neginf=1.0)
    return dataset

# Sanitize predictions and test dataset
predictions = sanitize_data(predictions)
test_dataset = sanitize_data(test_dataset)

# Check for invalid values in predictions
print("Predictions contain NaN values:", np.isnan(predictions.labels).any(), np.isnan(predictions.scores).any())
print("Test dataset contain NaN values:", np.isnan(test_dataset.labels).any(), np.isnan(test_dataset.scores).any())

# Debugging information
print("Sanitized predictions labels:", np.unique(predictions.labels, return_counts=True))
print("Sanitized predictions scores:", np.unique(predictions.scores, return_counts=True))
print("Sanitized test dataset labels:", np.unique(test_dataset.labels, return_counts=True))

# Function to ensure there are no invalid values
def check_invalid_values(dataset):
    print("Checking for invalid values...")
    for attr in ['labels', 'scores', 'instance_weights']:
        attr_values = getattr(dataset, attr)
        if np.isnan(attr_values).any() or np.isinf(attr_values).any():
            raise ValueError(f"Invalid values found in {attr}: NaN or Inf detected")

# Check for invalid values before postprocessing
check_invalid_values(predictions)
check_invalid_values(test_dataset)


Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.


epoch 0; iter: 0; batch classifier loss: 0.651670; batch adversarial loss: 0.617080
epoch 1; iter: 0; batch classifier loss: 0.457401; batch adversarial loss: 0.689148
epoch 2; iter: 0; batch classifier loss: 0.460568; batch adversarial loss: 0.679667
epoch 3; iter: 0; batch classifier loss: 0.557965; batch adversarial loss: 0.716553
epoch 4; iter: 0; batch classifier loss: 0.323063; batch adversarial loss: 0.612048
epoch 5; iter: 0; batch classifier loss: 0.260240; batch adversarial loss: 0.629764
epoch 6; iter: 0; batch classifier loss: 0.306680; batch adversarial loss: 0.620981
epoch 7; iter: 0; batch classifier loss: 0.440385; batch adversarial loss: 0.667346
epoch 8; iter: 0; batch classifier loss: 0.339268; batch adversarial loss: 0.619484
epoch 9; iter: 0; batch classifier loss: 0.199815; batch adversarial loss: 0.587002
epoch 10; iter: 0; batch classifier loss: 0.379700; batch adversarial loss: 0.634311
epoch 11; iter: 0; batch classifier loss: 0.324369; batch adversarial loss:

## Complete